# Day 3 Project: Build Your Own Regional Forecast Notebook

**SAMOS Operational Oceanography — Practical Tutorial, Day 3**

If you use Colab instead of your local computer do File → Save a copy in Drive, right after opening the Colab link, so your changes are saved.

On Day 1 you downloaded and mapped **Copernicus Marine (CMEMS)** forecast data. On Day 2 you explored **SOMISANA/CROCO** regional forecasts and compared different model configurations. Today you'll combine those skills to build your **own** short analysis notebook.

## Your task

Pick **one region** and produce a short notebook that:

1. **Downloads at least one *new* type of operational model data** — i.e. something you have not already downloaded in Day 1 or Day 2. For example:
   - A different CMEMS product: biogeochemistry (chlorophyll, oxygen, nutrients), waves, sea level anomaly, sea ice, an ocean reanalysis, ...
   - If you want to look at longer periods, you can choose to download monthly data.
   - You can download SOMISANA data - if it is in your region
   - Use the techniques from `Searching_CMEMS.ipynb` (`copernicusmarine.describe()`) to find a product, dataset, and variable that interests you.
     
2. **Produces at least one *new* type of plot** — something beyond the maps you already made on Day 1/2. For example:
   - A **Hovmöller diagram** (time vs. depth, or time vs. longitude/latitude, with colour = variable).
   - A **vertical section / transect** (distance along a line vs. depth).
   - A **scatter plot comparing model output to observations** (e.g. satellite or in-situ data vs. the forecast).
3. **Writes a short interpretation** of what you found (a short paragraph in a markdown cell is enough).  You can submit the notebook to me to get feedback.

This notebook is a **skeleton** — it has the setup you already know, plus signposted `# TODO` sections for the new parts you need to fill in. Delete/replace the section(s) you don't use, and feel free to reorganise once you have a plan.

## Suggested workflow

1. Decide on a region and a scientific question you're curious about (e.g. "how does chlorophyll vary along the coast?", "what does the thermocline look like across the shelf?", "how well does the forecast match a buoy/satellite observation?").
2. Pick the data source(s) that let you answer it (Section 3).
3. Download, inspect, and plot (Sections 4–6).
4. Write a short conclusion (Section 7).


## 1. Setup

In [ ]:
# Run once per environment (e.g. once in Colab, or once in your local conda/venv)
%pip install -q copernicusmarine xarray matplotlib cartopy netCDF4 pandas


In [ ]:
import os
from datetime import datetime, timedelta, timezone
from getpass import getpass

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import copernicusmarine

%matplotlib inline
plt.rcParams["figure.dpi"] = 100


## 2. Choose your region

Define a bounding box for the region you want to study — reuse a region from Day 1, or pick a new one anywhere in the world. Fill in the `TODO` values below.


In [ ]:
# TODO: set your own bounding box and give your region a name
REGION = {
    "name": "TODO: my region",
    "lon_min": 0.0,   # TODO
    "lon_max": 0.0,   # TODO
    "lat_min": 0.0,   # TODO
    "lat_max": 0.0,   # TODO
}


def preview_region(region):
    '''Draw the selected bounding box on a world map, as a sanity check.'''
    fig = plt.figure(figsize=(6, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    pad = 10
    ax.set_extent([
        region["lon_min"] - pad, region["lon_max"] + pad,
        region["lat_min"] - pad, region["lat_max"] + pad,
    ], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.gridlines(draw_labels=True)
    ax.plot(
        [region["lon_min"], region["lon_max"], region["lon_max"], region["lon_min"], region["lon_min"]],
        [region["lat_min"], region["lat_min"], region["lat_max"], region["lat_max"], region["lat_min"]],
        color="red", linewidth=2, transform=ccrs.PlateCarree(),
    )
    ax.set_title(f"Selected region: {region['name']}")
    plt.show()


preview_region(REGION)


## 3. Find and download a new type of data

Use `copernicusmarine.describe()` (see `Searching_CMEMS.ipynb`) to find a **product, dataset, and variable(s) you have not used before** — for example a biogeochemical, wave, sea-level, or sea-ice product. You can also mix in a SOMISANA file from Day 2 if it's useful for your question.

First, log in (same as Day 1):


In [ ]:
os.environ["CM_USERNAME"] = input("Copernicus Marine username: ")
os.environ["CM_PASSWORD"] = getpass("Copernicus Marine password: ")

copernicusmarine.login(
    username=os.environ["CM_USERNAME"],
    password=os.environ["CM_PASSWORD"],
    check_credentials_valid=True,
)


In [ ]:
# TODO: explore the catalogue to find a new product/dataset/variable.  
# You can use the code below or use the CMEMS Viewer from Monday.
# Example starting point (uncomment and adapt):
#
# catalogue = copernicusmarine.describe(
#     product_id="GLOBAL_ANALYSISFORECAST_BGC_001_028",  # TODO: pick a product
#     disable_progress_bar=True,
# )
# for dataset in catalogue.products[0].datasets:
#     print(dataset.dataset_id)


In [ ]:
# TODO: download your chosen variable(s) for your region, following the
# copernicusmarine.subset() pattern from 01_copernicus_marine_forecast_download.ipynb

# DATASET_ID = "TODO"
# VARIABLES = ["TODO"]

today = datetime.now(timezone.utc).date()
start_datetime = today.isoformat()
end_datetime = (today + timedelta(days=5)).isoformat()

output_dir = "cmems_data"
os.makedirs(output_dir, exist_ok=True)

# copernicusmarine.subset(
#     dataset_id=DATASET_ID,
#     username=os.environ["CM_USERNAME"],
#     password=os.environ["CM_PASSWORD"],
#     variables=VARIABLES,
#     minimum_longitude=REGION["lon_min"],
#     maximum_longitude=REGION["lon_max"],
#     minimum_latitude=REGION["lat_min"],
#     maximum_latitude=REGION["lat_max"],
#     start_datetime=start_datetime,
#     end_datetime=end_datetime,
#     output_directory=output_dir,
#     output_filename="my_new_variable.nc",
# )


## 4. Load and inspect the data

Open the file(s) you downloaded and look at their structure — dimensions, coordinates, variables, units, and time range — the same way you did on Day 1/2.


In [ ]:
# TODO: open your downloaded file(s)
# ds = xr.open_dataset(os.path.join(output_dir, "my_new_variable.nc"), engine="netcdf4")
# ds


## 5. A first map

Before building your new plot type, make one familiar map of your new variable (like Day 1/2) as a sanity check — does it look physically reasonable for your region?


In [ ]:
# TODO: a quick pcolormesh map of your new variable at one time step, e.g.:
#
# fig = plt.figure(figsize=(7, 7))
# ax = plt.axes(projection=ccrs.PlateCarree())
# ds["VARIABLE"].isel(time=0).plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree())
# ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=2)
# ax.add_feature(cfeature.COASTLINE, zorder=2)
# ax.gridlines(draw_labels=True)
# plt.show()


## 6. Build a new type of plot

Pick **at least one** of the three options below (or propose your own) and fill in the `TODO`s. Each is sketched as a starting point, not a finished solution — the exact `.sel()`/`.isel()` calls will depend on your dataset's dimension names and your chosen point/line.

### Option A — Hovmöller diagram (time vs. depth, or time vs. space)

Useful for seeing how a profile or a section evolves over the forecast period.


In [ ]:
# Option A: Hovmöller diagram
#
# TODO: pick a single (lon, lat) point (or a fixed latitude, letting longitude vary instead of depth)
# point = ds["VARIABLE"].sel(longitude=LON, latitude=LAT, method="nearest")
#
# # time vs. depth Hovmoller:
# fig, ax = plt.subplots(figsize=(8, 5))
# point.T.plot(ax=ax, x="time", y="depth", cmap="viridis")
# ax.invert_yaxis()
# ax.set_title("Hovmoller diagram: TODO variable/location")
# plt.show()


### Option B — Vertical section / transect

Useful for looking across a front, shelf break, or upwelling zone in cross-section.


In [ ]:
# Option B: vertical section along a fixed latitude (or longitude)
#
# TODO: pick a fixed latitude and a longitude range that crosses a feature you're interested in
# section = ds["VARIABLE"].isel(time=0).sel(latitude=LAT, method="nearest")
#
# fig, ax = plt.subplots(figsize=(8, 5))
# section.plot(ax=ax, x="longitude", y="depth", cmap="viridis")
# ax.invert_yaxis()
# ax.set_title("Section at latitude TODO")
# plt.show()


### Option C — Scatter plot: model vs. observations

You'll need an observational dataset that overlaps your region and time window in space and time — for example a moored buoy, an Argo float profile, or a satellite product (e.g. via the CMEMS **in-situ TAC**, or another Copernicus/NOAA/ERDDAP source). Use `Searching_CMEMS.ipynb`-style catalogue browsing to find one.


In [ ]:
# Option C: scatter plot of model vs. observations
#
# TODO: load your observation data (however it's provided — CSV, NetCDF, ERDDAP, ...)
# obs = ...  # e.g. a pandas DataFrame or xarray Dataset with matching lon/lat/time
#
# TODO: sample the model at the same locations/times as the observations, e.g.
# model_at_obs = ds["VARIABLE"].sel(
#     longitude=obs["lon"], latitude=obs["lat"], time=obs["time"], method="nearest"
# )
#
# fig, ax = plt.subplots(figsize=(5, 5))
# ax.scatter(obs["value"], model_at_obs, s=15, alpha=0.7)
# lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
# ax.plot(lims, lims, "k--", linewidth=1, label="1:1")
# ax.set_xlabel("Observed")
# ax.set_ylabel("Model")
# ax.legend()
# ax.set_title("Model vs. observations: TODO variable")
# plt.show()


## 7. Interpretation

*(Replace this cell with your own writing — a few sentences is enough.)*

- What did you download, for which region, and why?
- What does your new plot show? Does it match your expectations?
- If you compared to observations: how well does the forecast do? Where does it disagree most?
- What would you look at next if you had more time?


## Useful references

- Copernicus Marine Service catalogue: <https://data.marine.copernicus.eu>
- `copernicusmarine` toolbox documentation: <https://help.marine.copernicus.eu/en/collections/9080063-copernicus-marine-toolbox>
- SOMISANA catalogue: <https://catalog.somisana.ac.za/catalog/>
- `Searching_CMEMS.ipynb` — how to browse the CMEMS catalogue for new products/datasets/variables
- `01_copernicus_marine_forecast_download.ipynb` / `02_SOMISANA_forecast_tutorial.ipynb` — Day 1/2 tutorials this project builds on
